# 02 — Temporal Split and Feature Engineering

This notebook transforms the cleaned AML transactions into modelling-ready datasets.

Main steps:
- Define a leakage-safe temporal train/validation/test split
- Separate the anomalous September 11–18 period
- Engineer transaction, temporal, behavioural and relational features
- Save the resulting datasets for modelling

In [1]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.auto import tqdm

In [ ]:
# Project configuration

PROJECT_ROOT = Path("/content/drive/MyDrive/temporal-graph-aml")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

KAGGLE_DATASET = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"

RAW_TRANSACTION_FILE_NAME = "HI-Small_Trans.csv"
RAW_TRANSACTION_PATH = RAW_DIR / RAW_TRANSACTION_FILE_NAME

CLEAN_TRANSACTIONS_PATH = PROCESSED_DIR / "hi_small_transactions_clean.parquet"

LABEL_COLUMN = "is_laundering"

# Change this to True only if you want to rebuild the clean Parquet from CSV
FORCE_REBUILD_CLEAN_DATA = False

In [3]:
from google.colab import drive

drive.mount("/content/drive")

for folder in [
    RAW_DIR,
    PROCESSED_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder: {RAW_DIR}")
print(f"Processed data folder: {PROCESSED_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/temporal-graph-aml
Raw data folder: /content/drive/MyDrive/temporal-graph-aml/data/raw
Processed data folder: /content/drive/MyDrive/temporal-graph-aml/data/processed


In [4]:
transactions = pl.read_parquet(CLEAN_TRANSACTIONS_PATH)

print("Shape:", transactions.shape)
transactions.head()

Shape: (5078345, 15)


timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,sender_id,receiver_id,is_self_transfer,is_cross_currency
datetime[μs],i32,str,i32,str,f32,str,f32,str,str,i8,str,str,i8,i8
2022-09-01 00:00:00,3209,"""8000F4670""",3209,"""8000F4670""",14675.570312,"""US Dollar""",14675.570312,"""US Dollar""","""Reinvestment""",0,"""3209_8000F4670""","""3209_8000F4670""",1,0
2022-09-01 00:00:00,1420,"""8005DFEB0""",1420,"""8005DFEB0""",897.369995,"""US Dollar""",897.369995,"""US Dollar""","""Reinvestment""",0,"""1420_8005DFEB0""","""1420_8005DFEB0""",1,0
2022-09-01 00:00:00,10,"""8000F6850""",10,"""8000F6850""",99986.9375,"""US Dollar""",99986.9375,"""US Dollar""","""Reinvestment""",0,"""10_8000F6850""","""10_8000F6850""",1,0
2022-09-01 00:00:00,12,"""8001167D0""",12,"""8001167D0""",16.08,"""US Dollar""",16.08,"""US Dollar""","""Reinvestment""",0,"""12_8001167D0""","""12_8001167D0""",1,0
2022-09-01 00:00:00,1,"""8001364A0""",1,"""8001364A0""",10.3,"""US Dollar""",10.3,"""US Dollar""","""Reinvestment""",0,"""1_8001364A0""","""1_8001364A0""",1,0


In [5]:
transactions.select(
    pl.col("timestamp").min().alias("start_time"),
    pl.col("timestamp").max().alias("end_time"),
)

start_time,end_time
datetime[μs],datetime[μs]
2022-09-01 00:00:00,2022-09-18 16:18:00


In [6]:
transactions = transactions.unique(
    maintain_order=True
)

print("Shape after removing duplicates:", transactions.shape)

Shape after removing duplicates: (5078336, 15)


In [7]:
TRAIN_END=datetime(2022,9,8)
VAL_END=datetime(2022,9,10)
TEST_END=datetime(2022,9,11)

transactions =transactions.with_columns(
        pl.when(pl.col("timestamp")<TRAIN_END)
        .then(pl.lit("Train"))
        .when(pl.col("timestamp")<VAL_END)
        .then(pl.lit("Val"))
        .when(pl.col("timestamp")<TEST_END)
        .then(pl.lit("Test"))
        .otherwise(pl.lit("Special"))
        .alias("split")
)

split_summary = (
    transactions
    .group_by("split")
    .agg(
        pl.len().alias("n_transactions"),
        pl.col("is_laundering").sum().alias("n_laundering")
        )
    .with_columns(
        (pl.col("n_transactions")-pl.col("n_laundering")).alias("n_normal"),
        ((pl.col("n_laundering")/pl.col("n_transactions"))*100).alias("laundering_ratio")
    )
)

print(split_summary.to_pandas().to_string(index=False))

  split  n_transactions  n_laundering  n_normal  laundering_ratio
    Val         1137234          1053   1136181          0.092593
Special            1108           655       453         59.115523
  Train         3731669          3027   3728642          0.081117
   Test          208325           442    207883          0.212168


In [ ]:
transactions = transactions.with_columns(
    pl.col("timestamp").dt.hour().alias("hour"),
    (pl.col("timestamp").dt.weekday()>=6).alias("is_weekend")
)


In [16]:
transactions = transactions.with_columns(
    pl.col("amount_paid").log1p().alias("log_amount_paid"),
    pl.concat_str([pl.col("payment_currency"),pl.col("receiving_currency")],separator="_").alias("currency_pair")
)

In [17]:
transactions.select(
    "payment_currency",
    "receiving_currency",
    "currency_pair"
).head()

payment_currency,receiving_currency,currency_pair
str,str,str
"""US Dollar""","""US Dollar""","""US Dollar_US Dollar"""
"""US Dollar""","""US Dollar""","""US Dollar_US Dollar"""
"""US Dollar""","""US Dollar""","""US Dollar_US Dollar"""
"""US Dollar""","""US Dollar""","""US Dollar_US Dollar"""
"""US Dollar""","""US Dollar""","""US Dollar_US Dollar"""


In [ ]:
tran